# LOSO Evaluation — BSPC 2026 Major Revision

**Purpose**: Leave-One-Site-Out (LOSO) cross-validation to address expected reviewer concern about site/scanner generalization.

## Scientific rationale

The main paper uses 5-fold stratified CV which does **not** guarantee site-independence — subjects from the same acquisition site may appear in both training and test folds. LOSO directly tests whether the β-VAE + logistic regression pipeline generalizes across acquisition sites.

## Key design constraints

| Constraint | Value |
|---|---|
| Manufacturer filter | **Philips only** (CN subjects are exclusively Philips; cross-manufacturer LOSO is not feasible) |
| Held-out sites | **Primary**: [6, 18, 19, 130, 305] · **Strict**: [6, 130] |
| VAE pool | All Philips subjects (CN + AD + **MCI**) from non-held-out sites |
| Classifier train | Philips **CN/AD** from non-held-out sites |
| Classifier test | Philips **CN/AD** from held-out site |
| Leakage check | Hard `AssertionError` if any held-out subject appears in any training split |
| Primary classifier | Logistic regression (matches main paper + downstream interpretability) |

## Site breakdown (Philips-only)

| Site | CN | AD | Set |
|------|----|----|---------|
| 6 | 9 | 5 | primary + strict |
| 18 | 6 | 4 | primary only |
| 19 | 4 | 8 | primary only |
| 130 | 20 | 13 | primary + strict |
| 305 | 4 | 3 | primary only |

All 5 eligible sites use **Philips** scanners exclusively. Cross-manufacturer LOSO is not feasible (GE/SIEMENS have 0 CN subjects).

## Outputs

- `results/revision_bspc_2026/loso_primary/` — 5-site LOSO
- `results/revision_bspc_2026/loso_strict/` — 2-site LOSO (sites 6 and 130)

Each output directory contains per-site subdirectories and pooled metrics.

## Shared path configuration

All cells below use these variables. Edit here if paths change.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("/home/diego/proyectos/vae_AD").resolve()

SCRIPT = PROJECT_ROOT / "scripts" / "revision_bspc_2026" / "run_loso_cv.py"
TENSOR_PATH = PROJECT_ROOT / "data" / "AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned" / "GLOBAL_TENSOR_from_AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned.npz"
METADATA_PATH = PROJECT_ROOT / "data" / "SubjectsData_AAL3_procesado2.csv"

#TENSOR_PATH = (
#    "data/AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_"
#    "GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned/"
#    "GLOBAL_TENSOR_from_AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_"
#    "GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned.npz"
#)
#METADATA_PATH = "data/SubjectsData_AAL3_procesado2.csv"

# Channels matching baseline best run: indices 1, 0, 2
# = Pearson_OMST_GCE_Signed_Weighted, Pearson_Full_FisherZ_Signed, MI_KNN_Symmetric
CHANNELS = "1 0 2"
#SCRIPT = "scripts/revision_bspc_2026/run_loso_cv.py"

---
## Smoke test — single held-out site (site 130)

Run this first to verify the pipeline before committing to a full multi-site run.
Site 130 is the largest fold (20 CN + 13 AD held out). Expected runtime: ~45–90 min on GPU.

Uses reduced epochs (`--epochs_vae 256 --early_stopping_patience_vae 30`) for a fast sanity check.

In [2]:
!python {SCRIPT} \
  --global_tensor_path {TENSOR_PATH} \
  --metadata_path {METADATA_PATH} \
  --output_dir results/revision_bspc_2026/loso_smoketest \
  --channels_to_use {CHANNELS} \
  --loso_mode custom \
  --loso_sites 130 \
  --classifier_types logreg \
  --classifier_calibrate \
  --classifier_use_class_weight \
  --inner_folds 5 \
  --epochs_vae 256 \
  --vae_val_split_ratio 0.2 \
  --early_stopping_patience_vae 30 \
  --cyclical_beta_n_cycles 4 \
  --lr_scheduler_type cosine_warm \
  --lr_scheduler_T0 80 \
  --lr_scheduler_eta_min 5e-7 \
  --weight_decay_vae 5e-7 \
  --batch_size 64 \
  --beta_vae 6.5 \
  --dropout_rate_vae 0.15 \
  --latent_dim 256 \
  --n_jobs_gridsearch 8 \
  --save_fold_artefacts \
  --save_vae_training_history \
  --gridsearch_scoring roc_auc \
  --metadata_features Age Sex \
  --seed 42

LOSO mode: custom | Held-out sites: [130]
Manufacturer filter: Philips
Loading data...
Cargando tensor global desde: /home/diego/proyectos/vae_AD/data/AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned/GLOBAL_TENSOR_from_AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned.npz
Se leyeron 131 ROIs de 'roi_names_in_order'.
Se leyeron 131 etiquetas de red de 'network_labels_in_order'.
Tensor global cargado. Forma: (431, 7, 131, 131)
Cargando metadatos desde: /home/diego/proyectos/vae_AD/data/SubjectsData_AAL3_procesado2.csv
Metadatos cargados. Forma: (434, 33)
Channels selected: ['Pearson_Full_FisherZ_Signed', 'Pearson_OMST_GCE_Signed_Weighted', 'MI_KNN_Symmetric']
run_config.json saved to results/revision_bspc_2026/loso_smoketest
Device: cuda

  LOSO FOLD: site_130
  [site_130] TEST: 33 (CN=20, AD=13)
  [site_130] TRAIN/DEV: 102 CN/

---
## Primary LOSO analysis — 5 sites [6, 18, 19, 130, 305]

Full run with all baseline hyperparameters.
Expected runtime: ~5–10 hours on GPU (5 VAE trainings × ~2560 epochs).

This is the **primary result** for the BSPC revision (PA.1 / PA.4 in reviewer_action_matrix.md).

In [3]:
!python {SCRIPT} \
  --global_tensor_path {TENSOR_PATH} \
  --metadata_path {METADATA_PATH} \
  --output_dir results/revision_bspc_2026/loso_primary \
  --channels_to_use {CHANNELS} \
  --loso_mode primary \
  --classifier_types logreg \
  --classifier_calibrate \
  --classifier_use_class_weight \
  --inner_folds 5 \
  --epochs_vae 2560 \
  --vae_val_split_ratio 0.2 \
  --early_stopping_patience_vae 240 \
  --cyclical_beta_n_cycles 32 \
  --lr_scheduler_type cosine_warm \
  --lr_scheduler_T0 80 \
  --lr_scheduler_eta_min 5e-7 \
  --weight_decay_vae 5e-7 \
  --batch_size 64 \
  --beta_vae 6.5 \
  --dropout_rate_vae 0.15 \
  --latent_dim 256 \
  --n_jobs_gridsearch 8 \
  --save_fold_artefacts \
  --save_vae_training_history \
  --qc_check_scanner_leakage \
  --gridsearch_scoring roc_auc \
  --metadata_features Age Sex \
  --seed 42

LOSO mode: primary | Held-out sites: [6, 18, 19, 130, 305]
Manufacturer filter: Philips
Loading data...
Cargando tensor global desde: /home/diego/proyectos/vae_AD/data/AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned/GLOBAL_TENSOR_from_AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned.npz
Se leyeron 131 ROIs de 'roi_names_in_order'.
Se leyeron 131 etiquetas de red de 'network_labels_in_order'.
Tensor global cargado. Forma: (431, 7, 131, 131)
Cargando metadatos desde: /home/diego/proyectos/vae_AD/data/SubjectsData_AAL3_procesado2.csv
Metadatos cargados. Forma: (434, 33)
Channels selected: ['Pearson_Full_FisherZ_Signed', 'Pearson_OMST_GCE_Signed_Weighted', 'MI_KNN_Symmetric']
run_config.json saved to results/revision_bspc_2026/loso_primary
Device: cuda

  LOSO FOLD: site_006
  [site_006] TEST: 14 (CN=9, AD=5)
  [site_006] TRAIN

---
## Strict LOSO analysis — 2 sites [6, 130]

Complementary analysis using only the sites meeting the main ≥5/class threshold.
This provides the most conservative estimate of site generalization.

- Site 6:   9 CN, 5 AD held out
- Site 130: 20 CN, 13 AD held out

In [4]:
!python {SCRIPT} \
  --global_tensor_path {TENSOR_PATH} \
  --metadata_path {METADATA_PATH} \
  --output_dir results/revision_bspc_2026/loso_strict \
  --channels_to_use {CHANNELS} \
  --loso_mode strict \
  --classifier_types logreg \
  --classifier_calibrate \
  --classifier_use_class_weight \
  --inner_folds 5 \
  --epochs_vae 2560 \
  --vae_val_split_ratio 0.2 \
  --early_stopping_patience_vae 240 \
  --cyclical_beta_n_cycles 32 \
  --lr_scheduler_type cosine_warm \
  --lr_scheduler_T0 80 \
  --lr_scheduler_eta_min 5e-7 \
  --weight_decay_vae 5e-7 \
  --batch_size 64 \
  --beta_vae 6.5 \
  --dropout_rate_vae 0.15 \
  --latent_dim 256 \
  --n_jobs_gridsearch 8 \
  --save_fold_artefacts \
  --save_vae_training_history \
  --qc_check_scanner_leakage \
  --gridsearch_scoring roc_auc \
  --metadata_features Age Sex \
  --seed 42

LOSO mode: strict | Held-out sites: [6, 130]
Manufacturer filter: Philips
Loading data...
Cargando tensor global desde: /home/diego/proyectos/vae_AD/data/AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned/GLOBAL_TENSOR_from_AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned.npz
Se leyeron 131 ROIs de 'roi_names_in_order'.
Se leyeron 131 etiquetas de red de 'network_labels_in_order'.
Tensor global cargado. Forma: (431, 7, 131, 131)
Cargando metadatos desde: /home/diego/proyectos/vae_AD/data/SubjectsData_AAL3_procesado2.csv
Metadatos cargados. Forma: (434, 33)
Channels selected: ['Pearson_Full_FisherZ_Signed', 'Pearson_OMST_GCE_Signed_Weighted', 'MI_KNN_Symmetric']
run_config.json saved to results/revision_bspc_2026/loso_strict
Device: cuda

  LOSO FOLD: site_006
  [site_006] TEST: 14 (CN=9, AD=5)
  [site_006] TRAIN/DEV: 121 CN/AD

---
## Optional: SVM sensitivity analysis

Run after logreg to compare classifier robustness. Add `logreg` to `--classifier_types` to generate both in one run.

In [5]:
!python {SCRIPT} \
  --global_tensor_path {TENSOR_PATH} \
  --metadata_path {METADATA_PATH} \
  --output_dir results/revision_bspc_2026/loso_primary_svm \
  --channels_to_use {CHANNELS} \
  --loso_mode primary \
  --classifier_types svm logreg \
  --classifier_calibrate \
  --classifier_use_class_weight \
  --inner_folds 5 \
  --epochs_vae 2560 \
  --vae_val_split_ratio 0.2 \
  --early_stopping_patience_vae 240 \
  --cyclical_beta_n_cycles 32 \
  --lr_scheduler_type cosine_warm \
  --lr_scheduler_T0 80 \
  --lr_scheduler_eta_min 5e-7 \
  --weight_decay_vae 5e-7 \
  --batch_size 64 \
  --beta_vae 6.5 \
  --dropout_rate_vae 0.15 \
  --latent_dim 256 \
  --n_jobs_gridsearch 8 \
  --save_fold_artefacts \
  --save_vae_training_history \
  --gridsearch_scoring roc_auc \
  --metadata_features Age Sex \
  --seed 42

LOSO mode: primary | Held-out sites: [6, 18, 19, 130, 305]
Manufacturer filter: Philips
Loading data...
Cargando tensor global desde: /home/diego/proyectos/vae_AD/data/AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned/GLOBAL_TENSOR_from_AAL3_dynamicROIs_fmri_tensor_NeuroEnhanced_v6.5.17_AAL3_131ROIs_OMST_GCE_Signed_GrangerLag1_ChNorm_ROIreorderedYeo17_ParallelTuned.npz
Se leyeron 131 ROIs de 'roi_names_in_order'.
Se leyeron 131 etiquetas de red de 'network_labels_in_order'.
Tensor global cargado. Forma: (431, 7, 131, 131)
Cargando metadatos desde: /home/diego/proyectos/vae_AD/data/SubjectsData_AAL3_procesado2.csv
Metadatos cargados. Forma: (434, 33)
Channels selected: ['Pearson_Full_FisherZ_Signed', 'Pearson_OMST_GCE_Signed_Weighted', 'MI_KNN_Symmetric']
run_config.json saved to results/revision_bspc_2026/loso_primary_svm
Device: cuda

  LOSO FOLD: site_006
  [site_006] TEST: 14 (CN=9, AD=5)
  [site_006] T